# DINOv3 Dense Feature Distillation

DINOv3 ViT-S/16のdense featureを少数画像で診断し、全件cache、E004a Baseline、E004b feature KDを順に実行します。先に `docs/dinov3-feature-distillation.md` を読んでください。

In [ ]:
import subprocess
import sys
from pathlib import Path


def run(*arguments: str) -> None:
    subprocess.run([sys.executable, "-m", "fm_to_edge_seg", *arguments], check=True)


run("doctor")

`cuda_available: True`を確認し、承認後に取得したweightのファイル名へ変更します。

In [ ]:
MANIFEST = Path("data/deepcrack/manifest.csv")
REPOSITORY = Path("external/dinov3")
WEIGHTS = Path("external/dinov3_weights/dinov3_vits16_pretrain_lvd1689m.pth")
DEBUG_CACHE = Path("teacher_cache/dinov3_vits16_debug")
FULL_CACHE = Path("teacher_cache/dinov3_vits16_544x384")

## 1. 3画像のfeature cache

In [ ]:
run(
    "create-dinov3-feature-cache",
    str(MANIFEST),
    str(DEBUG_CACHE),
    "--repository",
    str(REPOSITORY),
    "--weights",
    str(WEIGHTS),
    "--width",
    "544",
    "--height",
    "384",
    "--max-samples",
    "3",
)

In [ ]:
run(
    "preview-feature-cache",
    str(MANIFEST),
    str(DEBUG_CACHE),
    "artifacts/dinov3_vits16_debug.png",
    "--limit",
    "3",
)

PCA画像に空間構造があり、全面同色・ノイズ・位置ずれでないことを確認してから次へ進みます。

## 2. train全件のfeature cache

In [ ]:
run(
    "create-dinov3-feature-cache",
    str(MANIFEST),
    str(FULL_CACHE),
    "--repository",
    str(REPOSITORY),
    "--weights",
    str(WEIGHTS),
    "--width",
    "544",
    "--height",
    "384",
)

## 3. 公平なA/B学習
E004aとE004bはfeature KD以外の条件を揃えています。

In [ ]:
run("train", "configs/experiment/e004a_aligned_aug_baseline.yaml")

In [ ]:
run("train", "configs/experiment/e004b_dinov3_feature_kd.yaml")